In [41]:
import json
import pandas as pd
import requests

In [42]:
series_dict = {
    "CUUR0000SAF1": "Food",
    "CUUR0000SAH": "Housing",
    "CUUR0000SAE": "Education",
    "CUUR0000SAT": "Transportation",
    "CUUR0000SAM": "Medical",
    "CUUR0000SA0E": "Energy",
}

In [43]:
url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
headers = {"Content-type": "application/json"}

In [ ]:
payload = json.dumps(
    {
        "seriesid": list(series_dict.keys()),
        "startyear": "2017",
        "endyear": "2026"
    }
)

In [45]:
response = requests.post(url, data=payload, headers=headers)
json_data = response.json()

In [ ]:
all_records = []

if json_data.get("status") == "REQUEST_SUCCEEDED":
    for series in json_data["Results"]["series"]:
        series_id = series["seriesID"]
        category_name = series_dict[series_id]

        for row in series["data"]:
            year = row["year"]
            period = row["period"]  
            raw_value = row["value"]

            if raw_value == "-":
                value = None 
            else:
                value = float(raw_value)

            month = period.replace("M", "")
            if month == "13":
                continue

            date_str = f"{year}-{month}-01"

            all_records.append(
                {
                    "Date": pd.to_datetime(date_str),
                    "Category": category_name,
                    "CPI_Index": value,
                }
            )
else:
    print("API Error:", json_data.get("message"))

In [ ]:
if all_records:
    df = pd.DataFrame(all_records)

    df_pivoted = df.pivot(
        index="Date", columns="Category", values="CPI_Index"
    ).sort_index()

    output_filename = "cpi_data.csv"
    df_pivoted.to_csv(output_filename)

    print(f"Success! Data has been exported and saved as: {output_filename}")

    

Success! Data has been exported and saved as: cpi_data.csv
